# Advance Seeds — Train a Run

## ▶ To start: click **Runtime → Run all** (or press `Cmd/Ctrl + F9`)

Colab does **not** execute a notebook automatically when it opens — even when launched from the dashboard's *Open in Colab* button. You have to trigger the run yourself once. After that, sit back: each cell streams output, including live training metrics back to the registry dashboard.

---

This notebook trains the YOLO model for a registry run id and streams metrics + the final artifact back to Supabase.

**Before you Run all, check:**
1. **GPU runtime is set**: Runtime → Change runtime type → T4 (free) or better.
2. **You have a Supabase service-role key** — the notebook will prompt for it.
3. **Run id is in the URL** as `?run_id=...`. If not, paste it manually when the notebook prompts.

## 1. Resolve the run id from the URL

In [ ]:
from urllib.parse import urlparse, parse_qs
RUN_ID = ''
try:
    from google.colab import _message
    url = _message.blocking_request('get_url', timeout_sec=5)
    qs = parse_qs(urlparse(url).query)
    RUN_ID = qs.get('run_id', [''])[0]
except Exception as exc:
    print('Could not auto-read run_id from the URL:', exc)
print('Run id:', RUN_ID or '(none — paste below)')

In [ ]:
RUN_ID = RUN_ID or input('Run id: ').strip()
assert RUN_ID, 'A run id is required.'

## 2. Install dependencies + clone the repo

In [ ]:
%pip install -q ultralytics supabase
!git clone --depth 1 https://github.com/phongsakorn-ipassion/advance-seeds-field-inspector-ml.git
%cd advance-seeds-field-inspector-ml
%pip install -q -e .

## 3. Configure Supabase access

Paste your **service-role** key (not anon). It only lives in this notebook session.

In [ ]:
import getpass, os
os.environ['SUPABASE_URL'] = 'https://gqsxiohxokgwwugeoxmy.supabase.co'
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = getpass.getpass('SUPABASE_SERVICE_ROLE_KEY: ')
os.environ['ADVANCE_SEEDS_RUN_ID'] = RUN_ID

## 4. Pull run config + dataset

In [ ]:
from supabase import create_client
sb = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SERVICE_ROLE_KEY'])
run = sb.table('runs').select('*').eq('id', RUN_ID).single().execute().data
config = run['config_yaml']
print('dataset:', config.get('dataset'))
print('source_weights:', config.get('source_weights'))
print('classes:', len(config.get('classes', [])))
print('hyperparameters:', config.get('hyperparameters'))

In [ ]:
DATASET = config.get('dataset', '')
if DATASET.startswith('datasets/'):
    print('TODO: fetch', DATASET, 'from R2 and place at the path YOLO expects')
else:
    print('Using local dataset reference:', DATASET)

## 5. Train + stream metrics back

The Python SDK (`src/advance_seeds_ml/registry/`) writes per-epoch metrics to the `run_metrics` table during training, which appear live in the dashboard's Live tracking panel.

In [ ]:
!python scripts/train_yolo26n_seg.py \
    --run-id $ADVANCE_SEEDS_RUN_ID \
    --report-registry

## 6. Done

When the run finishes, switch back to the dashboard — the candidate version, final metrics, and the R2 artifact should all be visible. Promote it to staging or production from the Models screen.